In [1]:
%run 05_functions_for_modelling.ipynb

In [2]:
df = pd.read_csv('hsls_clean.csv')

with open('train_test_split.json') as f:
    split = json.load(f)

with open('feature_tiers.json') as f:
    feature_tiers = json.load(f)

train_idx = split['train']
test_idx  = split['test']

# Tier 1 — Coarse Search

In [3]:
tier1_search, X_train_tier1, y_train_tier1, X_test_tier1, y_test_tier1 = fit_models(
    features=feature_tiers['tier1'], df=df, train_idx=train_idx, test_idx=test_idx)

print("Best params:", tier1_search.best_params_)
print("Best CV F1:", round(tier1_search.best_score_, 4))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params: {'subsample': 0.6, 'n_estimators': 750, 'max_depth': 7, 'learning_rate': 0.005, 'colsample_bytree': 0.7}
Best CV F1: 0.5137


# Tier 1 — Fine Search

In [ ]:
fine_grid_tier1 = define_fine_grid(tier1_search.cv_results_)
tier1_fine_search = fit_fine_search(fine_grid_tier1, X_train_tier1, y_train_tier1)

Score range: 0.4477 — 0.5137
Threshold: 0.5005 (16 combinations above threshold)

    mean_test_score  std_test_score  param_n_estimators  param_max_depth  param_learning_rate  param_subsample  param_colsample_bytree
4          0.513746        0.008695                 750                7                0.005              0.6                     0.7
75         0.513169        0.010957                1000                6                0.005              0.6                     1.0
89         0.512398        0.010279                 100                8                0.050              0.8                     0.7
88         0.509692        0.007781                 300                7                0.010              0.7                     0.9
8          0.508961        0.010500                1000                7                0.005              1.0                     0.8
56         0.508059        0.020297                 200               10                0.005              0

# Tier 2 — Coarse Search

In [ ]:
tier2_search, X_train_tier2, y_train_tier2, X_test_tier2, y_test_tier2 = fit_models(
    features=feature_tiers['tier2'], df=df, train_idx=train_idx, test_idx=test_idx)

print("Best params:", tier2_search.best_params_)
print("Best CV F1:", round(tier2_search.best_score_, 4))

# Tier 2 — Fine Search

In [ ]:
fine_grid_tier2 = define_fine_grid(tier2_search.cv_results_)
tier2_fine_search = fit_fine_search(fine_grid_tier2, X_train_tier2, y_train_tier2)

# Tier 3 — Coarse Search

In [ ]:
tier3_search, X_train_tier3, y_train_tier3, X_test_tier3, y_test_tier3 = fit_models(
    features=feature_tiers['tier3'], df=df, train_idx=train_idx, test_idx=test_idx)

print("Best params:", tier3_search.best_params_)
print("Best CV F1:", round(tier3_search.best_score_, 4))

# Tier 3 — Fine Search

In [ ]:
fine_grid_tier3 = define_fine_grid(tier3_search.cv_results_)
tier3_fine_search = fit_fine_search(fine_grid_tier3, X_train_tier3, y_train_tier3)

# Tier 4 — Coarse Search

In [ ]:
tier4_search, X_train_tier4, y_train_tier4, X_test_tier4, y_test_tier4 = fit_models(
    features=feature_tiers['tier4'], df=df, train_idx=train_idx, test_idx=test_idx)

print("Best params:", tier4_search.best_params_)
print("Best CV F1:", round(tier4_search.best_score_, 4))

# Tier 4 — Fine Search

In [ ]:
fine_grid_tier4 = define_fine_grid(tier4_search.cv_results_)
tier4_fine_search = fit_fine_search(fine_grid_tier4, X_train_tier4, y_train_tier4)

# Evaluation of all models

In [ ]:
evaluate_model(tier1_fine_search, X_train_tier1, y_train_tier1, X_test_tier1, y_test_tier1, tier=1)
evaluate_model(tier2_fine_search, X_train_tier2, y_train_tier2, X_test_tier2, y_test_tier2, tier=2)
evaluate_model(tier3_fine_search, X_train_tier3, y_train_tier3, X_test_tier3, y_test_tier3, tier=3)
evaluate_model(tier4_fine_search, X_train_tier4, y_train_tier4, X_test_tier4, y_test_tier4, tier=4)

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, accuracy_score

summary = []

for tier, fine_search, X_train, y_train, X_test, y_test in [
    (1, tier1_fine_search, X_train_tier1, y_train_tier1, X_test_tier1, y_test_tier1),
    (2, tier2_fine_search, X_train_tier2, y_train_tier2, X_test_tier2, y_test_tier2),
    (3, tier3_fine_search, X_train_tier3, y_train_tier3, X_test_tier3, y_test_tier3),
    (4, tier4_fine_search, X_train_tier4, y_train_tier4, X_test_tier4, y_test_tier4),
]:
    model = fine_search.best_estimator_
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    summary.append({
        'Tier': tier,
        'Features': X_test.shape[1],
        'CV F1': round(fine_search.best_score_, 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Test Precision': round(precision_score(y_test, y_pred), 4),
        'Test Recall': round(recall_score(y_test, y_pred), 4),
        'Test F1': round(f1_score(y_test, y_pred), 4),
        'Test ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))